In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="j-hartmann/emotion-english-distilroberta-base",
    local_dir="./emotion_model",
    allow_patterns=[
        "config.json",
        "*.safetensors",        # or pytorch_model.bin if no safetensors file exists
        "pytorch_model.bin",
        "vocab.json",
        "merges.txt",
        "tokenizer.json",
        "tokenizer_config.json",
        "special_tokens_map.json",
    ],
)

c:\Users\LENOVO\OneDrive\Desktop\Ai agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 11.39it/s]


'C:\\Users\\LENOVO\\OneDrive\\Desktop\\Ai agent\\Emotion_model\\emotion_model'

In [2]:
import re
from dotenv import load_dotenv
load_dotenv()
from typing import TypedDict
from transformers import pipeline
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel,Field

In [1]:
emotion_model = pipeline(
    "text-classification",
    model="./emotion_model",   # local folder instead of the repo name
    top_k=None,
    device=-1,
)

NameError: name 'pipeline' is not defined

In [8]:
SENTIMENT_MAP  = {
    # Positive
    "joy": "positive",

    # Neutral / ambiguous
    "surprise": "neutral",
    "neutral": "neutral",

    # Negative
    "sadness": "negative",
    "anger": "negative",
    "fear": "negative",
    "disgust": "negative",
}

INTENSIFIERS = {"very", "so", "really", "extremely", "totally", "absolutely", "literally"}
URGENT_WORDS = {"urgent", "asap", "immediately", "now", "emergency", "quickly", "hurry"}
SLANG = {"lol", "bro", "dude", "gonna", "wanna", "pls", "plz", "omg", "btw", "idk"}

In [9]:
class Tone(BaseModel):
    formality: str
    urgency: str
    is_task: bool = Field(default=False, description="identifies if the message is user ans to do any task or not")
    polite: bool
class Emotion_sentiment_intensity_Tone(BaseModel):
    emotion: str
    sentiment: str
    intensity: float
    tone: Tone

In [10]:
class Detect_emotion:
    def get_emotion_sentiment_intensity_tone(test: str):
        output = emotion_model(test)
        max_emoptions = max(output[0], key=lambda x: x['score'])
        return max_emoptions
    
    def get_sentiment(emotion: str):
        return SENTIMENT_MAP.get(emotion, "unknown")
    
    def get_intensity(text: str, emotion_conf: float) -> float:
        words = re.findall(r"[a-zA-Z']+", text)
        if not words:
            return 0.0
        caps = sum(1 for w in words if w.isupper() and len(w) > 2) / len(words)
        exclaim = min(text.count("!"), 3) / 3
        repeats = 1.0 if re.search(r"(.)\1{2,}", text) else 0.0   # "sooooo"
        boosters = min(sum(w.lower() in INTENSIFIERS for w in words), 2) / 2

        score = (0.4 * emotion_conf + 0.2 * caps + 0.2 * exclaim
                + 0.1 * repeats + 0.1 * boosters)
        return round(min(score, 1.0), 2)
    
    def get_style(text: str) -> dict:
        words = set(re.findall(r"[a-z']+", text))
        casual = bool(words & SLANG) or bool(re.search(r"[\U0001F300-\U0001FAFF]", text)) or text.islower()
        return {
            "formality": "casual" if casual else "formal",
            "urgency": "urgent" if (words & URGENT_WORDS) else "relaxed",
            "is_task": "?" in text,
            "polite": bool(words & {"please", "thanks", "thank", "kindly", "appreciate"}),
            }

In [11]:
def get_emotion_sentiment_intensity_tone(text: str) -> Emotion_sentiment_intensity_Tone:
    emotion_data = Detect_emotion.get_emotion_sentiment_intensity_tone(text)
    sentiment = Detect_emotion.get_sentiment(emotion = emotion_data['label'])
    intensity = Detect_emotion.get_intensity(text, emotion_data['score'])
    style = Detect_emotion.get_style(text)
    
    return {
        "emotion": emotion_data['label'],
        "sentiment": sentiment,
        "intensity": intensity,
        "tone": style
    } # pyright: ignore[reportReturnType]